In [ ]:
# pip install yfinance pandas

import yfinance as yf
import pandas as pd

tickers = [
    "XOM", "CVX", "COP", "OXY",
    "SLB", "HAL", "EOG", "DVN", "PLC", "MPC", "FANG",
    "PSX", "VLO", "CTRA", "APA"
]

start_date = "2020-01-01"
end_date = "2026-04-10"

raw = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    group_by="ticker",
    progress=False
)

dfs = []

for ticker in tickers:
    temp = raw[ticker].copy()
    temp.columns = [col.lower().replace(" ", "_") for col in temp.columns]
    temp["ticker"] = ticker
    dfs.append(temp)

df = pd.concat(dfs)

df = df.reset_index()
df = df.rename(columns={"Date": "date"})

# Keep same features
features = [
    "date", "ticker",
    "open", "high", "low", "close",
    "adj_close", "volume"
]

df = df[features]

# Optional: sort
df = df.sort_values(["date", "ticker"]).reset_index(drop=True)

df.head()

,date,ticker,open,high,low,close,adj_close,volume
0,2020-01-02,APA,25.700001,25.930000,25.129999,25.360001,21.528528,3395100.0
1,2020-01-02,COP,65.279999,65.680000,64.849998,65.459999,52.157879,4122800.0
2,2020-01-02,CTRA,17.520000,17.600000,16.930000,17.230000,12.906062,8088300.0
3,2020-01-02,CVX,120.809998,121.629997,120.769997,121.430000,92.017723,5205000.0
4,2020-01-02,DVN,26.209999,26.280001,25.629999,25.790001,18.640133,5091400.0


In [ ]:
# Keep only dates where all tickers have data
counts = df.groupby("date")["ticker"].nunique()
valid_dates = counts[counts == len(tickers)].index

df_aligned = df[df["date"].isin(valid_dates)].copy()

df_aligned = df_aligned.sort_values(["date", "ticker"]).reset_index(drop=True)

df_aligned.head()

,date,ticker,open,high,low,close,adj_close,volume
0,2020-01-02,APA,25.700001,25.930000,25.129999,25.360001,21.528528,3395100.0
1,2020-01-02,COP,65.279999,65.680000,64.849998,65.459999,52.157879,4122800.0
2,2020-01-02,CTRA,17.520000,17.600000,16.930000,17.230000,12.906062,8088300.0
3,2020-01-02,CVX,120.809998,121.629997,120.769997,121.430000,92.017723,5205000.0
4,2020-01-02,DVN,26.209999,26.280001,25.629999,25.790001,18.640133,5091400.0


In [ ]:
# Save aligned price data to the current directory.
df_aligned.to_csv("oil_gas_stocks_aligned.csv", index=False)
